In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
from datasets import load_dataset

def load_nq_german(data_file = "./data/ng_german.jsonl.gz"):
    # Load the JSONL file as a dataset
    dataset = (
        load_dataset("json", data_files=data_file, split="train",num_proc=8)
        .remove_columns(["query", "answer"])
        .rename_column("question_de", "query")
        .rename_column("answer_de", "answer")
    )
    dataset_dict = dataset.train_test_split(test_size=1_000, seed=12)
    return dataset_dict


/home/ogu/source/workshop-ki-deepdive/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import logging
import random

import numpy
import torch
#from torch import mps  # noqa: F401
#torch.mps.device = mps
from datasets import Dataset

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerModelCardData,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss, CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

/tmp/ipykernel_493460/1351605734.py:16: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers.losses import MultipleNegativesRankingLoss, CachedMultipleNegativesRankingLoss
/tmp/ipykernel_493460/1351605734.py:17: DeprecationWarning: Importing from 'sentence_transformers.training_args' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.training_args' instead.
  from sentence_transformers.training_args import BatchSamplers


In [4]:
logging.basicConfig(format="%(asctime)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S", level=logging.INFO)
random.seed(12)
torch.manual_seed(12)
numpy.random.seed(12)

In [5]:
# Feel free to adjust these variables:
use_prompts = True
include_prompts_in_pooling = True

# 1. Load a model to finetune with 2. (Optional) model card data
base_model_name = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"

In [6]:
model = SentenceTransformer(
    base_model_name,
    #tokenizer_kwargs={"max_seq_length": 512},
    model_card_data=SentenceTransformerModelCardData(
        language="de",
        license="apache-2.0",
        model_name=f"{base_model_name} trained on german Natural Questions pairs",
    ),
).to(torch.bfloat16)

2026-09-04 16:17:52 - No device provided, using cuda:0
2026-09-04 16:17:53 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-09-04 16:17:53 - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-04 16:17:53 - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/4328cf26390c98c5e3c738b4460a05b95f4911f5/modules.json?%2Fsentence-transformers%2Fparaphrase-multilingual-mpnet-base-v2%2Fresolve%2Fmain%2Fmodules.json=&etag=%22f7640f94e81bb7f4f04daf1668850b38763a13d9%22 "HTTP/1.1 200 OK"
2026-09-04 16:17:53 - HTTP Request: HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-09-04 16:17:53 - HTT

In [7]:
model.set_pooling_include_prompt(include_prompts_in_pooling)

In [8]:
# 2. (Optional) Define prompts
if use_prompts:
    query_prompt = "query: "
    corpus_prompt = "document: "
    prompts = {
        "query": query_prompt,
        "answer": corpus_prompt,
    }

In [9]:
# 3. Load a dataset to finetune on
dataset_dict = load_nq_german()
train_dataset: Dataset = dataset_dict["train"]
eval_dataset: Dataset = dataset_dict["test"]


2026-09-04 16:17:57 - HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/json/json.py "HTTP/1.1 200 OK"


In [10]:
# 4. Define a loss function
loss = CachedMultipleNegativesRankingLoss(model, mini_batch_size=16) 
#loss = MultipleNegativesRankingLoss(model) # <- this does work with mps (Apple Silicon)


In [11]:
# 5. (Optional) Specify training arguments

# todo: limit train size for testing

run_name = "nq-german-" + base_model_name.split("/")[-1]
if use_prompts:
    run_name += "-prompts"
if not include_prompts_in_pooling:
    run_name += "-exclude-pooling-prompts"
args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir=f"models/{run_name}",
    # Optional training parameters:
    num_train_epochs=0.25, # limit training to 1/4 epoch for development
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    learning_rate=4e-5,
    warmup_ratio=0.1,   
    fp16=False,  # Set to False if you get an error that your GPU can't run on FP16
    bf16=True,  # Set to True if you have a GPU that supports BF16
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # MultipleNegativesRankingLoss benefits from no duplicate samples in a batch
    # Optional tracking/debugging parameters:
    eval_strategy="steps",
    eval_steps=0.5,
    save_strategy="steps",
    save_steps=0.5,
    save_total_limit=2,
    logging_steps=5,
    logging_first_step=True,
    run_name=run_name,  # Will be used in W&B if `wandb` is installed
    seed=12,
    prompts=prompts if use_prompts else None,
)

2026-09-04 16:17:57 - The `warmup_ratio` argument is deprecated in Transformers v5+, and will also be removed from Sentence Transformers once support for Transformers v4 is dropped. Since you're using Transformers v5+, please use `warmup_steps` (as a float) to specify the warmup ratio instead.


In [12]:
# 7. Create a trainer & train
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
    #evaluator=dev_evaluator,
)
trainer.train()

Step,Training Loss,Validation Loss
97,0.051900,0.043916
194,0.061328,0.042049


2026-09-04 16:18:24 - Saving model checkpoint to models/nq-german-paraphrase-multilingual-mpnet-base-v2-prompts/checkpoint-97
2026-09-04 16:18:24 - Saving model to models/nq-german-paraphrase-multilingual-mpnet-base-v2-prompts/checkpoint-97
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.14it/s]
2026-09-04 16:18:51 - Saving model checkpoint to models/nq-german-paraphrase-multilingual-mpnet-base-v2-prompts/checkpoint-194
2026-09-04 16:18:51 - Saving model to models/nq-german-paraphrase-multilingual-mpnet-base-v2-prompts/checkpoint-194
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.34it/s]


TrainOutput(global_step=194, training_loss=0.06833847819529858, metrics={'train_runtime': 55.2185, 'train_samples_per_second': 449.07, 'train_steps_per_second': 3.513, 'total_flos': 0.0, 'train_loss': 0.06833847819529858, 'epoch': 0.2503225806451613})

In [16]:
# 8. Save the trained model
model.save_pretrained(f"models/{run_name}/final")

2026-09-04 16:21:15 - Saving model to models/nq-german-paraphrase-multilingual-mpnet-base-v2-prompts/final
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]


In [17]:
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer(f"./models/{run_name}/final")

# The sentences to encode
sentences = [
    "query: Das ist eine Frage über Kekse.",
    "answer: Das hier ist ein Keksrezept",
    "query: Ich mag Möven, oder?",
    "answer: Ich bin Paul die Möve",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)


similarities = model.similarity(embeddings, embeddings)
print(similarities)

2026-09-04 16:21:17 - No device provided, using cuda:0
2026-09-04 16:21:17 - Loading SentenceTransformer model from ./models/nq-german-paraphrase-multilingual-mpnet-base-v2-prompts/final.
Batches: 100%|██████████| 1/1 [00:00<00:00, 190.78it/s]

tensor([[1.0000, 0.8457, 0.2337, 0.0958],
        [0.8457, 1.0000, 0.2299, 0.2066],
        [0.2337, 0.2299, 1.0000, 0.3376],
        [0.0958, 0.2066, 0.3376, 1.0000]])


In [18]:
import numpy as np
# convert files for tensorflow embedding projector

# Save as TSV
np.savetxt('output.tsv', embeddings, delimiter='\t', fmt='%g')

with open("description.tsv","w") as f:
    f.writelines(sentences)